![banner](https://github.com/hello-robot/stretch_mujoco/raw/main/docs/images/stretch_mujoco.png)

<h1><center>Getting Started Tutorial  <a href="https://colab.research.google.com/github/hello-robot/stretch_mujoco/blob/main/docs/getting_started.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

This notebook provides an introduction to Stretch's [**MuJoCo** simulation](https://github.com/hello-robot/stretch_mujoco/). You can run this notebook with either **CPU** or **GPU** instance.


## Install


In [ ]:
# # Set up Stretch Mujoco repo
# # %rm -rf ./stretch_mujoco/
# # # !git clone https://github.com/hello-robot/stretch_mujoco --recurse-submodules
# # !git clone https://github.com/hello-robot/stretch_mujoco --depth 1
# # %cd ./stretch_mujoco/
# # %pip install -e ".[jupyter]"

# # Check if we can use GPU rendering
# import os
# import subprocess
# try:
#     subprocess.run('nvidia-smi')
#     USE_GPU=True
# except:
#     USE_GPU=False

# # Setup rendering
# if USE_GPU:
#     # Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
#     # This is usually installed as part of an Nvidia driver package, but the Colab
#     # kernel doesn't install its driver via APT, and as a result the ICD is missing.
#     # (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
#     NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
#     if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
#         with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
#             f.write("""{
#       "file_format_version" : "1.0.0",
#       "ICD" : {
#           "library_path" : "libEGL_nvidia.so.0"
#       }
#   }""")

#     # Configure MuJoCo to use the EGL rendering backend (requires GPU)
#     print('Setting environment variable to use GPU rendering:')
#     %env MUJOCO_GL=egl
# else:
#     # Required for OSMesa OpenGL driver
#     !apt-get update
#     !apt-get install -y libosmesa6-dev libgl1-mesa-glx libglfw3

#     print('Setting environment variable to use CPU rendering:')
#     %env MUJOCO_GL=osmesa

# # Other imports and helper functions
# import time
# import pprint
# import itertools
# import numpy as np

# # Graphics and plotting.
# !command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
# !pip install -q mediapy
# import mediapy as media
# import matplotlib.pyplot as plt

# # More legible printing from numpy.
# np.set_printoptions(precision=3, suppress=True, linewidth=100)

After installing, we need robocasa set up


In [ ]:
# python -m pip install -e ".[robocasa]"
# python -m pip install -e third_party/robosuite
# python -m pip install -e third_party/robocasa

If you get an error like

(stretch_mujoco_v310) orrijoa@orrijoa-IdeaPad-Gaming-3-15ACH6:~/projects/stretch_mujoco_jupyter/stretch_mujoco$ python -m pip install -e third_party/robosuite
Obtaining file:///home/orrijoa/projects/stretch_mujoco_jupyter/stretch_mujoco/third_party/robosuite
ERROR: file:///home/orrijoa/projects/stretch_mujoco_jupyter/stretch_mujoco/third_party/robosuite does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

From the root of your stretch_mujoco repo, run:
git submodule update --init


In [ ]:
# python third_party/robosuite/robosuite/scripts/setup_macros.py
# python third_party/robocasa/robocasa/scripts/setup_macros.py
# python third_party/robocasa/robocasa/scripts/download_kitchen_assets.py

## Basics - SETTINNG UP PARTS


The `StretchMujocoSimulator` class is used to:

- Start/stop the simulation
- Read camera imagery
- Read lidar scans
- Read joint states
- Position control the robot's ranged joints
- Velocity control the robot's mobile base


### Prerequisites to run to stream cameras in each windows


In [ ]:
# Viewer on NVIDIA dGPU (windowed)
import os
os.environ.pop("MUJOCO_GL", None)              # ensure not headless
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"

# If you want headless instead, comment the above and use:
# import os, shutil
# os.environ["MUJOCO_GL"] = "egl" if shutil.which("nvidia-smi") else "osmesa"

import time, threading, math
from pprint import pprint

import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

# Optional dependency
try:
    import mediapy as media
    HAS_MEDIAPY = True
except ImportError:
    HAS_MEDIAPY = False
    print("mediapy not installed. Install with: python -m pip install mediapy")
    
# Project imports last (after env is set!)
from stretch_mujoco import StretchMujocoSimulator
from stretch_mujoco.enums.stretch_sensors import StretchSensors
from stretch_mujoco.enums.stretch_cameras import StretchCameras
from stretch_mujoco.enums.actuators import Actuators

### Camera & Streaming Set Up


In [ ]:
# keep track of which windows we've created
_created_windows = set()

def show_camera_feeds_sync(sim, print_fps=False, init_size=(640, 480)):
    """
    Pull camera data from the simulator and display it using OpenCV.
    Windows are resizable (drag corners).
    """
    camera_data = sim.pull_camera_data()

    if print_fps:
        s = sim.pull_status()
        print(f"Physics fps: {s.fps}. Camera FPS: {camera_data.fps}. {s.sim_to_real_time_ratio_msg}")

    for cam_enum, pixels in camera_data.get_all(use_depth_color_map=True).items():
        if pixels is None:
            continue

        name = cam_enum.name  # window title

        # create a resizable window once per camera
        if name not in _created_windows:
            cv2.namedWindow(name, cv2.WINDOW_NORMAL)       # <-- resizable
            cv2.resizeWindow(name, *init_size)             # initial size
            # optional: keep aspect ratio when resizing
            try:
                cv2.setWindowProperty(name, cv2.WND_PROP_ASPECT_RATIO, cv2.WINDOW_KEEPRATIO)
            except Exception:
                pass
            _created_windows.add(name)

        cv2.imshow(name, pixels)

    # process window events (needed for resizing to take effect)
    cv2.waitKey(1)
    
_stream_evt = None
_stream_thread = None

def start_stream_thread(sim, print_fps=False, target_hz=20):
    """Run camera streaming in the background using your show_camera_feeds_sync()."""
    global _stream_evt, _stream_thread
    if _stream_thread and _stream_thread.is_alive():
        print("Stream already running.")
        return

    _stream_evt = threading.Event()

    def _worker():
        try:
            dt = 1.0 / max(1, target_hz)
            while not _stream_evt.is_set() and sim.is_running():
                show_camera_feeds_sync(sim, print_fps)
                # sim.step()                       # advance physics
                time.sleep(dt)                   # throttle display FPS a bit
        except Exception as e:
            print("stream thread ended:", type(e).__name__, e)
        finally:
            # Close any OpenCV windows cleanly
            try:
                cv2.destroyAllWindows()
            except: 
                pass

    _stream_thread = threading.Thread(target=_worker, daemon=True)
    _stream_thread.start()
    print("Stream thread started.")

def stop_stream_thread():
    """Stop the background stream cleanly (call before sim.stop())."""
    global _stream_evt, _stream_thread
    if _stream_evt:
        _stream_evt.set()
    if _stream_thread:
        _stream_thread.join(timeout=2.0)
    try:
        cv2.waitKey(1)
        cv2.destroyAllWindows()
    except:
        pass
    _stream_evt = None
    _stream_thread = None
    print("Stream thread stopped.")


### Robocasa Set up


In [ ]:
# Core libs
import mujoco
import numpy as np

# RoboCasa generator
try:
    from stretch_mujoco.robocasa_gen import model_generation_wizard
    print("Found model_generation_wizard()")
except Exception as e:
    print("Could not import model_generation_wizard:", e)

# robosuite / robocasa sanity
import robosuite
print("robosuite version:", getattr(robosuite, "__version__", "unknown"))

import robocasa
print("robocasa version:", getattr(robocasa, "__version__", "unknown"))

# Show robosuite macro backend if present
try:
    from robosuite import macros as RS_MACROS
    print("robosuite MUJOCO_GL:", getattr(RS_MACROS, "MUJOCO_GL", "not set"))
except Exception as e:
    print("robosuite macros import issue:", e)


### Mujoco Env Set Up


In [ ]:
import math

# your measured base pose
x = 0.8713510702553015
y = -1.3006141423127842
theta = 3.1147300993742104

# wrap theta to [-pi, pi] (optional but nice)
theta = math.atan2(math.sin(theta), math.cos(theta))

# Use the z from the default fixture pose you saw printed once (replace this!)
z0 = 0.0  # <-- replace with the third value from the printed "Adding stretch..." pos

w = math.cos(theta / 2.0)
z = math.sin(theta / 2.0)

robot_spawn_pose = {
    "pos": f"{x} {y} {z0}",
    "quat": f"{w} 0 0 {z}",
}

In [ ]:
mj_model = xml = objects_info = None

try:
    # Non interactive example. Adjust task/layout/style later if you want.
    # These names are common defaults; if they ever change, the except block will let you pick via wizard.
    mj_model, xml, objects_info = model_generation_wizard(
        task="PnPCounterToCab",
        layout=0,
        style=0,
        robot_spawn_pose=robot_spawn_pose,
    )
    print("Generated RoboCasa model non-interactively.")
except TypeError:
    # Some versions use only the interactive wizard
    print("Non-interactive args not supported. Opening interactive wizard...")
    mj_model, xml, objects_info = model_generation_wizard()
except FileNotFoundError as e:
    print("Asset missing:", e)
    print("Re-run the RoboCasa asset downloader script and try again.")
    raise
except Exception as e:
    print("RoboCasa scene generation failed:", e)
    raise

print("Model OK:", isinstance(mj_model, mujoco.MjModel))
if xml:
    print("XML length:", len(xml))
if objects_info is not None:
    # objects_info is usually a dict from the generator
    print("Objects info keys:", list(objects_info)[:5])


In [ ]:
# Pretty-print objects and their initial placements
for body_name, info in objects_info.items():
    print(f"{body_name:30s}  cat={info['cat']:12s}  pos={info['pos']}  quat={info['quat']}")

### Stretch Robot Start Set Up


In [ ]:
# sim = StretchMujocoSimulator(cameras_to_use=StretchCameras.all())

# cameras_to_use = [StretchCameras.cam_nav_rgb]
# cameras_to_use = StretchCameras.all()
cameras_to_use = []

sim = StretchMujocoSimulator(model=mj_model, cameras_to_use=cameras_to_use)
# sim.start(show_viewer_ui=False, headless=True)
sim.start(show_viewer_ui=False, headless=False)

### Helper Function For Model Training and Evaluation Set Up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stretch_reach_env import StretchReachEnv

class ProgressCallback(BaseCallback):
    def __init__(self, check_freq=1000, verbose=0):
        super().__init__(verbose)
        self.check_freq = check_freq
        self.episode_count = 0

    def _on_step(self) -> bool:
        # Count finished episodes
        dones = self.locals.get("dones")
        if dones is not None:
            self.episode_count += sum(dones)

        if self.num_timesteps % self.check_freq == 0:
            print(
                f"Steps: {self.num_timesteps} | "
                f"Episodes: {self.episode_count}"
            )
        return True

LOG_DIR = "./logs/stretch_reach"
os.makedirs(LOG_DIR, exist_ok=True)

def make_env(action_mode="discrete", allowed_parts="all", control_hold=3, run_name="run"):
    def _init():
        env = StretchReachEnv(
            sim,
            obj_name="apple0_main",
            dt=0.05,
            max_steps=100,          # Fetch-like
            success_thresh=0.06,
            
            action_mode = action_mode,
            allowed_parts = allowed_parts,
            
            wait_motion = True,
            control_hold = control_hold,
        )
        
        run_dir = os.path.join(LOG_DIR, run_name)
        os.makedirs(run_dir, exist_ok=True)
        
        env = Monitor(
            env,
            filename=os.path.join(run_dir, f"monitor_{action_mode}"),
            info_keywords=("is_success", "distance"),
        )
        return env
    return _init

def run_eval(model, env, n_steps=500, deterministic=True):
    reset_out = env.reset()
    obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

    dists = []
    episode_successes = 0
    episode_count = 0

    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)

        dists.append(info.get("distance", np.nan))
        if terminated or truncated:
            episode_count += 1
            episode_successes += int(info.get("is_success", 0.0) == 1.0)

            reset_out = env.reset()
            obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out
        
    mean_dist = float(np.nanmean(dists))
    success_rate = (episode_successes / max(1, episode_count))
    return mean_dist, episode_successes, episode_count, success_rate

def plot_monitor_progress(
    csv_path,
    smooth_window=20,
    show_reward=True,
    show_distance=True,
    show_success=True,
):
    """
    Plot training progress from Stable-Baselines3 Monitor CSV.

    Args:
        csv_path (str): Path to monitor CSV file.
        smooth_window (int): Moving average window size.
        show_reward (bool): Plot episode reward.
        show_distance (bool): Plot final episode distance (if exists).
        show_success (bool): Plot success rate (if exists).
    """

    # Load CSV (skip metadata row)
    df = pd.read_csv(csv_path, comment="#")

    episodes = range(len(df))

    # ---- Reward ----
    if show_reward and "r" in df.columns:
        plt.figure()
        plt.plot(episodes, df["r"])
        plt.xlabel("Episode")
        plt.ylabel("Episode Reward")
        plt.title("Reward per Episode")
        plt.show()

        # Smoothed reward
        df["reward_smooth"] = df["r"].rolling(window=smooth_window).mean()

        plt.figure()
        plt.plot(episodes, df["reward_smooth"])
        plt.xlabel("Episode")
        plt.ylabel(f"Smoothed Reward ({smooth_window})")
        plt.title("Smoothed Reward")
        plt.show()

    # ---- Distance ----
    if show_distance and "distance" in df.columns:
        plt.figure()
        plt.plot(episodes, df["distance"])
        plt.xlabel("Episode")
        plt.ylabel("Final Distance")
        plt.title("Distance per Episode")
        plt.show()

    # ---- Success Rate ----
    if show_success and "is_success" in df.columns:
        df["success_smooth"] = df["is_success"].rolling(window=smooth_window).mean()

        plt.figure()
        plt.plot(episodes, df["success_smooth"])
        plt.xlabel("Episode")
        plt.ylabel(f"Success Rate ({smooth_window})")
        plt.title("Success Rate (Moving Average)")
        plt.show()
        
# def run_eval_with_stuck(model, env, n_steps=500, deterministic=True):
#     reset_out = env.reset()
#     obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

#     dists = []
#     episode_successes = 0
#     episode_count = 0
#     stuck_steps = 0
#     total_steps = 0

#     for _ in range(n_steps):
#         action, _ = model.predict(obs, deterministic=deterministic)
#         obs, reward, terminated, truncated, info = env.step(action)

#         total_steps += 1
#         dists.append(info.get("distance", np.nan))
#         stuck_steps += int(bool(info.get("is_stuck", False)))

#         if terminated or truncated:
#             episode_count += 1
#             episode_successes += int(info.get("is_success", 0.0) == 1.0)

#             reset_out = env.reset()
#             obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

#     mean_dist = float(np.nanmean(dists))
#     success_rate = (episode_successes / max(1, episode_count))
#     stuck_rate = stuck_steps / max(1, total_steps)
#     return mean_dist, episode_successes, episode_count, success_rate, stuck_rate

# def run_eval_with_stuck_debug(model, env, n_steps=500, deterministic=True, near_dist=0.25, arm_attempt_thresh=0.20):
#     reset_out = env.reset()
#     obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

#     dists = []
#     episode_successes = 0
#     episode_count = 0

#     total_steps = 0
#     stuck_steps = 0

#     attempt_steps = 0
#     stuck_steps_given_attempt = 0

#     for _ in range(n_steps):
#         action, _ = model.predict(obs, deterministic=deterministic)
#         obs, reward, terminated, truncated, info = env.step(action)

#         total_steps += 1
#         d = info.get("distance", np.nan)
#         dists.append(d)

#         is_stuck = bool(info.get("is_stuck", False))
#         stuck_steps += int(is_stuck)

#         # "Attempt" guard: either close-ish OR clearly extending the arm.
#         arm_pos = float(info.get("arm_pos", 0.0))  # only works if env puts this in info
#         is_attempt = (np.isfinite(d) and d < near_dist) or (arm_pos > arm_attempt_thresh)

#         if is_attempt:
#             attempt_steps += 1
#             stuck_steps_given_attempt += int(is_stuck)

#         if terminated or truncated:
#             episode_count += 1
#             episode_successes += int(info.get("is_success", 0.0) == 1.0)
#             reset_out = env.reset()
#             obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

#     mean_dist = float(np.nanmean(dists))
#     success_rate = (episode_successes / max(1, episode_count))
#     stuck_rate = stuck_steps / max(1, total_steps)
#     attempt_rate = attempt_steps / max(1, total_steps)
#     stuck_given_attempt = stuck_steps_given_attempt / max(1, attempt_steps)

#     return mean_dist, episode_successes, episode_count, success_rate, stuck_rate, attempt_rate, stuck_given_attempt


## Manual ENV Testing Start


In [ ]:
%load_ext autoreload
%autoreload 2
from stretch_reach_env import StretchReachEnv

env = StretchReachEnv(
    sim,
    obj_name="apple0_main",
    dt=0.05,
    max_steps=100,
    success_thresh=0.06,
    
    action_mode = "discrete",
    allowed_parts = "lift_arm",
    
    wait_motion = True,
    control_hold = 3,
)

obs, info = env.reset()

print("reset info:", info)
print("obs keys:", obs.keys())
print("observation shape:", obs["observation"].shape, obs["observation"].dtype)
print("achieved_goal shape:", obs["achieved_goal"].shape, obs["achieved_goal"].dtype)
print("desired_goal shape:", obs["desired_goal"].shape, obs["desired_goal"].dtype)

In [ ]:
env.step(2)

In [ ]:
sim.set_object_pose("apple0_main", pos_xyz=(0.9, -0.5, 0.98), quat_wxyz=(1,0,0,0))

In [ ]:
sim.set_object_pose("apple0_main", pos_xyz=(0.9, -0.5, 1), quat_wxyz=(1,0,0,0))

In [ ]:
# for i in range(10):
#     # t_before = env.sim.pull_status().time
#     obs, r, term, trunc, info = env.step(4)
#     # t_after = env.sim.pull_status().time
#     # obs, r, term, trunc, info = env.step([1, 1, 1])
#     # obs, r, term, trunc, info = env.step([1, 0, 0])
    
#     print("=== from info ===")
#     print(f"i={i} lift pos/vel=({info['lift_pos']:.6f}, {info['lift_vel']:.6f}) "
#         f"arm pos/vel=({info['arm_pos']:.6f}, {info['arm_vel']:.6f}) "
#         f"grip pos/vel=({info['grip_pos']:.6f}, {info['grip_vel']:.6f})")
#     print(f"reward={r} ")
#     # print("sim time delta:", t_after - t_before)

In [ ]:
# # LIFT_STEP = [0.02, 0.04, 0.06, 0.08]
# LIFT_STEP = [0.04]

# for lift in LIFT_STEP:
#     env.LIFT_STEP = lift
#     for i in range(10):
#         # t_before = env.sim.pull_status().time
#         obs, r, term, trunc, info = env.step([0, 0, -1])
#         # t_after = env.sim.pull_status().time
#         # obs, r, term, trunc, info = env.step([1, 1, 1])
#         # obs, r, term, trunc, info = env.step([1, 0, 0])
        
#         print("=== from info ===")
#         print(f"i={i} lift pos/vel=({info['lift_pos']:.6f}, {info['lift_vel']:.6f}) "
#           f"arm pos/vel=({info['arm_pos']:.6f}, {info['arm_vel']:.6f}) "
#           f"grip pos/vel=({info['grip_pos']:.6f}, {info['grip_vel']:.6f})")
#         # print("sim time delta:", t_after - t_before)
        
#     # sim.move_to(Actuators.lift, 0)
#     # time.sleep(3)

## TRAINING THE MODEL Start

In [ ]:
# # NOTE: total_timesteps is the number of env.step() transitions collected.
# # Removing time.sleep() makes wall-clock faster but does NOT change what a "timestep" means here.

# TOTAL_TIMESTEPS = 1024  # per chunk
# NUM_CHUNKS = 20
# action_mode = "discrete"
# allowed_parts = "lift_arm"
# control_hold = 3

# run_name = f"{time.strftime('%Y%m%d_%H%M%S')}_{action_mode}_{allowed_parts}_k{control_hold}"

# LOG_DIR = "./logs/stretch_reach"
# os.makedirs(LOG_DIR, exist_ok=True)

# vec_env = DummyVecEnv([make_env(
#     action_mode=action_mode,
#     allowed_parts=allowed_parts,
#     control_hold=control_hold, 
#     run_name=run_name
#     )])

# model = PPO(
#     policy="MultiInputPolicy",
#     env=vec_env,
#     verbose=1,
#     n_steps=256,          # small, safe default
#     batch_size=64,
#     learning_rate=3e-4,
#     gamma=0.98,
#     tensorboard_log=LOG_DIR,
#  )

# progress_cb = ProgressCallback()

# for i in range(NUM_CHUNKS):
#     model.learn(
#         total_timesteps=TOTAL_TIMESTEPS,
#         reset_num_timesteps=(i == 0),  # first chunk starts fresh, then continue
#         callback=progress_cb,
#         tb_log_name=run_name,
#     )
#     model.save(f"ppo_stretch_reach_{model.num_timesteps}_{action_mode}_{allowed_parts}_k{control_hold}")
#     print(f"Saved at timestep {model.num_timesteps}")

### ADDITIONAL MODEL TRAINING

In [ ]:
# LOG_DIR = "./logs/stretch_reach"
# os.makedirs(LOG_DIR, exist_ok=True)

# TOTAL_TIMESTEPS = 1_024  # add more later
# NUM_CHUNKS = 20          # total training = 1024 * 10
# action_mode = "discrete"
# allowed_parts = "lift_arm"
# control_hold = 3

# # Your existing TB run folder name (the one already created)
# tb_run_name = "20260221_135539_discrete_lift_arm_k3"

# # Load model and tell it where TB logs live
# model_name = "ppo_stretch_reach_20480_discrete_lift_arm_k3"
# model = PPO.load(model_name, tensorboard_log=LOG_DIR)

# # IMPORTANT: make sure env spaces match the original training
# vec_env = DummyVecEnv([make_env(
#     action_mode=action_mode,
#     allowed_parts=allowed_parts,
#     control_hold=1,
#     run_name="continuing"
#     )])
# model.set_env(vec_env)

# progress_cb = ProgressCallback()

# for i in range(NUM_CHUNKS):
#     # Continue training and write into the SAME TB folder
#     model.learn(
#         total_timesteps=TOTAL_TIMESTEPS,
#         reset_num_timesteps=False,
#         callback=progress_cb,
#         tb_log_name=tb_run_name,
#     )
#     model.save(model_name)
#     print(f"Saved at timestep {model.num_timesteps}")
    

### LOAD PPO MODEL

In [ ]:
# model = PPO.load("ppo_stretch_reach_increased_max_steps_20000")
model = PPO.load("ppo_stretch_reach_40960_discrete_lift_arm_k3")

In [ ]:
# def action_delta_stats(env, model, n=200):
#     # returns avg delta distance for each discrete action index
#     counts = {}
#     sum_delta = {}
#     obs, _ = env.reset()
#     for _ in range(n):
#         action, _ = model.predict(obs, deterministic=True)
#         action_int = int(np.asarray(action).reshape(-1)[0])
#         s_before = env._get_status()
#         dist_before = float(env._compute_distance(env._get_obs()))
#         obs2, r, term, trunc, info = env.step(action)
#         dist_after = float(info["distance"])
#         delta = dist_before - dist_after  # positive means distance decreased (good)
#         counts[action_int] = counts.get(action_int, 0) + 1
#         sum_delta[action_int] = sum_delta.get(action_int, 0.0) + delta
#         if term or trunc:
#             obs, _ = env.reset()
#         else:
#             obs = obs2
#     stats = {a: {"count": counts.get(a, 0), "avg_delta": (sum_delta.get(a,0)/counts.get(a,1)) if counts.get(a,0) else 0.0}
#              for a in range(env.action_space.n)}
#     return stats

# # Usage (raw env, not DummyVecEnv)
# eval_env = StretchReachEnv(
#     sim,
#     obj_name="apple0_main",
#     dt=0.05,
#     max_steps=100,
#     success_thresh=0.06,
    
#     action_mode = "discrete",
#     allowed_parts = "lift_arm",
    
#     wait_motion = True,
#     control_hold = 3,
# )
# print(action_delta_stats(eval_env, model, n=500))

### SIMPLE MODEL EVALUATION

In [ ]:
action_mode = "discrete"
allowed_parts = "lift_arm"
control_hold = 3

eval_env = DummyVecEnv([make_env(
    action_mode=action_mode,
    allowed_parts=allowed_parts,
    control_hold=control_hold,
    run_name="eval"
    )])

# RESET FIRST
obs = eval_env.reset()

In [ ]:
# sim.set_object_pose("apple0_main", pos_xyz=(0.9, -0.2, 0.92), quat_wxyz=(1,0,0,0))

In [ ]:
for i in range(100):
    action, _ = model.predict(obs, deterministic=True)
    obs, rewards, dones, infos = eval_env.step(action)

    info = infos[0]   # because DummyVecEnv wraps it
    print(
        f"i={i}, "
        f"act={int(action)}, "
        f"rew={rewards[0]:.4f}, "
        f"delta_arm={info['delta_arm']:.8f}, "
        f"arm_pos={info['arm_pos']:.6f}"
    )

    if dones[0]:
        obs = eval_env.reset()

    time.sleep(0.02)

In [ ]:
eval_env.reset()

### GRAPH EVALUATION

In [ ]:
plot_monitor_progress("monitor_train.csv")

### TRAINING + EVALUATION TO TEST IMPROVEMENT GETTING STUCK AT THE TABLE LIP

In [ ]:
# vec_env = DummyVecEnv([make_env])

# model = PPO(
#     policy="MultiInputPolicy",
#     env=vec_env,
#     seed=SEED,
#     verbose=0,
#     n_steps=256,          # small, safe default
#     batch_size=64,
#     learning_rate=3e-4,
#     gamma=0.98,
#  )

# eval_env = StretchReachEnv(sim, obj_name="apple0_main", dt=0.05, max_steps=150, success_thresh=0.06)
# eval_env = Monitor(eval_env, info_keywords=("is_success", "distance"))

# TOTAL = 10_000
# CHUNK = 2_000

# for t in range(0, TOTAL, CHUNK):
#     model.learn(total_timesteps=CHUNK, reset_num_timesteps=False)

#     mean_dist, ep_succ, ep_cnt, succ_rate, stuck_rate = run_eval_with_stuck(
#         model, eval_env, n_steps=800, deterministic=True
#     )
#     print(
#         f"t={t+CHUNK:6d}  mean_dist={mean_dist:.4f}  succ={succ_rate:.1%}  "
#         f"episodes={ep_cnt}  stuck_rate={stuck_rate:.1%}"
#     )

#     # IMPORTANT: eval reset/home changed sim state; re-sync the training env
#     vec_env.reset()

# model.save(f"ppo_stretch_reach_improved_getting_stuck_{TOTAL}")

In [ ]:
# mean_dist, ep_succ, ep_cnt, succ_rate, stuck_rate, attempt_rate, stuck_given_attempt = run_eval_with_stuck_debug(
#     model, eval_env, n_steps=800, deterministic=True
# )
# print(
#     f"t={t+CHUNK:6d} mean_dist={mean_dist:.4f} succ={succ_rate:.1%} episodes={ep_cnt} "
#     f"stuck={stuck_rate:.1%} attempt={attempt_rate:.1%} stuck|attempt={stuck_given_attempt:.1%}"
# )

In [ ]:
# def probe_info_fields(model, env, n_steps=300, deterministic=True, near_dist=0.25, arm_attempt_thresh=0.20):
#     reset_out = env.reset()
#     obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

#     missing_is_stuck = 0
#     missing_arm_pos = 0

#     attempt_by_dist = 0
#     attempt_by_arm = 0
#     stuck_true = 0

#     min_d = float("inf")
#     frac_d_lt_0p15 = 0
#     frac_d_lt_0p10 = 0
#     frac_d_lt_0p06 = 0

#     for _ in range(n_steps):
#         action, _ = model.predict(obs, deterministic=deterministic)
#         obs, reward, terminated, truncated, info = env.step(action)

#         d = info.get("distance", np.nan)
#         if np.isfinite(d):
#             min_d = min(min_d, float(d))
#             frac_d_lt_0p15 += int(d < 0.15)
#             frac_d_lt_0p10 += int(d < 0.10)
#             frac_d_lt_0p06 += int(d < 0.06)

#         if "is_stuck" not in info:
#             missing_is_stuck += 1
#         stuck_true += int(bool(info.get("is_stuck", False)))

#         if "arm_pos" not in info:
#             missing_arm_pos += 1
#             arm_pos = 0.0
#         else:
#             arm_pos = float(info["arm_pos"])

#         by_dist = (np.isfinite(d) and d < near_dist)
#         by_arm = (arm_pos > arm_attempt_thresh)

#         attempt_by_dist += int(by_dist)
#         attempt_by_arm += int(by_arm)

#         if terminated or truncated:
#             reset_out = env.reset()
#             obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

#     n = max(1, n_steps)
#     print("missing is_stuck:", missing_is_stuck, "/", n)
#     print("missing arm_pos :", missing_arm_pos, "/", n)
#     print("attempt_by_dist :", attempt_by_dist / n)
#     print("attempt_by_arm  :", attempt_by_arm / n)
#     print("stuck_true_rate :", stuck_true / n)
#     print("min distance    :", min_d)
#     print("frac d<0.15     :", frac_d_lt_0p15 / n)
#     print("frac d<0.10     :", frac_d_lt_0p10 / n)
#     print("frac d<0.06     :", frac_d_lt_0p06 / n)

In [ ]:
# probe_info_fields(model, eval_env, n_steps=300)

### END SIMULATION

In [ ]:
# 1) Stop the current headless sim (if running)
if sim.is_running():
    sim.stop()

## TRAINING PART END - ETC PARTS AFTER

### GET OBJ STATE (APPLE)


In [ ]:
sim.register_tracked_objects(list(objects_info.keys()))

In [ ]:
obj_name = "apple0_main"

In [ ]:
obj_state = sim.pull_objects_state([obj_name])
print(obj_state)

In [ ]:
limits = {
    Actuators.lift: (0.0, 1.1),
    Actuators.arm:  (0.0, 0.52),
    Actuators.gripper: (-0.25, 0.53),
}

# how long we wait between actions.
dt = 0.05
# action scales (start small, tune later)
scales = np.array([0.03, 0.03, 0.02], dtype=np.float32)  # [lift, arm, gripper]

In [ ]:
def get_ee_pos(sim):
    T = sim.get_ee_pose()
    return T[:3, 3].astype(float)

def get_obj_pos(sim, obj_name):
    return sim.pull_objects_state()[obj_name]["pos"].astype(float)

def distance_to_object(sim, obj_name):
    ee_pos = get_ee_pos(sim)
    obj_pos = get_obj_pos(sim, obj_name)
    d = float(np.linalg.norm(ee_pos - obj_pos))
    return d, ee_pos, obj_pos

def manual_step(sim, a, obj_name):
    """
    a: array-like shape (3,), values in [-1, 1]
    returns: d, ee_pos, obj_pos
    """
    a = np.asarray(a, dtype=np.float32)
    a = np.clip(a, -1.0, 1.0) # a is just the policy saying “move up/down a bit” (a direction and strength).
    delta = a * scales # scales is shape (3,)

    s = sim.pull_status()
    current_lift = float(s.lift.pos)
    current_arm  = float(s.arm.pos)
    current_grip = float(s.gripper.pos)
    
    lift_low, lift_high = limits[Actuators.lift]
    arm_low,  arm_high  = limits[Actuators.arm]
    grip_low, grip_high = limits[Actuators.gripper]
    
    target_lift = np.clip(current_lift + float(delta[0]), lift_low, lift_high)
    target_arm  = np.clip(current_arm  + float(delta[1]), arm_low,  arm_high)
    target_grip = np.clip(current_grip + float(delta[2]), grip_low, grip_high)

    sim.move_to(Actuators.lift,    target_lift)
    sim.move_to(Actuators.arm,     target_arm)
    sim.move_to(Actuators.gripper, target_grip)

    # fixed control tick
    time.sleep(dt)

    return distance_to_object(sim, obj_name)


In [ ]:
distance_to_object(sim, obj_name)

In [ ]:
# move only lift up
for i in range(5):
    d, ee, obj = manual_step(sim, a=[1, 0, 0], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} lift={s.lift.pos:.6f} ee_z={ee[2]:.6f}")

In [ ]:
# move only arm forward
for i in range(5):
    d, ee, obj = manual_step(sim, a=[0, 1, 0], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} arm={s.arm.pos:.6f} ee_x={ee[0]:.6f} ee_z={ee[2]:.6f}")

In [ ]:
# move only gripper (try open)
for i in range(5):
    d, ee, obj = manual_step(sim, a=[0, 0, 1], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} gripper={s.gripper.pos:.6f}")


In [ ]:
# move only gripper (try close)
for i in range(5):
    d, ee, obj = manual_step(sim, a=[0, 0, -1], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} gripper={s.gripper.pos:.6f}")


In [ ]:
start_stream_thread(sim, print_fps=False, target_hz=20)

### ROBOT MANIPULATION


In [ ]:
sim.home()

In [ ]:
# from -2.02 to 0.49 (document) - tested use this value
# from -1.53 to 0.79 (current set up) - correct

sim.move_to(Actuators.head_tilt, 0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.head_tilt, 0.2)
time.sleep(0.5)

In [ ]:
# from -4.04 to 1.73
sim.move_to(Actuators.head_pan, 1.73)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.head_pan, -0.2)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.base_translate, 0.05)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.base_translate, -0.05)
time.sleep(0.5)

In [ ]:
# 90 degree ~= 1.57
sim.move_by(Actuators.base_rotate, 1.5)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.base_rotate, -0.1)
time.sleep(0.5)

In [ ]:
# from 0 to 1.1 - tested
sim.move_to(Actuators.lift, 0.7)
time.sleep(0.5)

In [ ]:
pprint(sim.pull_status().lift)

In [ ]:
for i in range(100):
    print("LIFT===================")
    pprint(sim.pull_status().lift)
    sim.move_by(Actuators.lift, 0.008)
    pprint(sim.pull_status().lift)

# sim.move_by(Actuators.lift, -0.000001)
# sim.wait_while_is_moving(Actuators.lift)
# sim.move_by(Actuators.lift, -0.2)

In [ ]:
pprint(sim.pull_status().lift)

In [ ]:
# from 0 to 0.13 (current)
# from 0 to 0.52 (document) - correct - tested
# sim.move_to(Actuators.arm, 0.0)
# time.sleep(0.5)

for i in range(100):
    print("LIFT===================")
    pprint(sim.pull_status().arm)
    sim.move_by(Actuators.arm, 0.000006)
    pprint(sim.pull_status().arm)

In [ ]:
sim.move_by(Actuators.arm, 0.05)
time.sleep(0.5)

In [ ]:
# (-1.39, 4.42) - tested
sim.move_to(Actuators.wrist_yaw, 0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.wrist_yaw, -0.2)
time.sleep(0.5)

In [ ]:
# (-1.57, 0.56) current - correct - tested
# (-1.57, 0.57) document
sim.move_to(Actuators.wrist_pitch, 0.0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.wrist_pitch, -0.2)
time.sleep(0.5)

In [ ]:
# (-3.14, 3.14)
sim.move_to(Actuators.wrist_roll, 0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.wrist_roll, -0.2)
time.sleep(0.5)

In [ ]:
# (-0.02, 0.04)
# (-0.3,0.55) - tested
sim.move_to(Actuators.gripper, 0)
time.sleep(0.5)

In [ ]:
# sim.move_by(Actuators.gripper, 0.05)
# time.sleep(0.5)

for i in range(10):
    print("LIFT===================")
    pprint(sim.pull_status().gripper)
    sim.move_by(Actuators.gripper, 0.004)
    pprint(sim.pull_status().gripper)

In [ ]:
# # Try to force it "very closed"
# sim.move_to(Actuators.gripper, -1.0)
# time.sleep(1)
# print("closed?", sim.pull_status().gripper.pos)

# # Try to force it "very open"
# sim.move_to(Actuators.gripper,  1.0)
# time.sleep(1)
# print("open?", sim.pull_status().gripper.pos)


### Report current status and limits


In [ ]:
pprint(sim.pull_status())

In [ ]:
pprint(sim.get_ee_pose())

In [ ]:
pprint(sim.register_tracked_objects(["apple0_main"]))

In [ ]:
pprint(sim.pull_all_objects_state())

In [ ]:
pprint(sim.pull_camera_data())



In [ ]:
# radar related
pprint(sim.pull_sensor_data())

In [ ]:
print(sim.pull_joint_limits())

### End the simulation


In [ ]:
# # Try to undo any previous manual wrapping if you still have the originals
# try:
#     sys.stdout = _sys_stdout_orig
#     sys.stderr = _sys_stderr_orig
# except NameError:
#     pass

# 1) stop background streaming first
stop_stream_thread()

# 2) give the sim process a short moment to finish its own threads
time.sleep(0.1)

# 3) stop the simulator
if sim.is_running():
    sim.stop()

# 4) as a final sweep, make sure no OpenCV windows remain
try:
    cv2.waitKey(1)
    cv2.destroyAllWindows()
except:
    pass
